# CKD Deepfake — End-to-End Pipeline

**Satu notebook, jalan semua.** Setelah setup awal (mount Drive, clone repo, install deps, download teachers & DF40), tinggal Run All → hasil akhir berupa student checkpoints, edge-eval JSON, dan figure PNG/PDF langsung tersimpan di Drive.

## Urutan

| Bagian | Isi | Kapan dijalankan |
|--------|-----|------------------|
| 1 | Mount Drive + clone repo + install | setiap session |
| 2 | Download teacher weights (EfficientNet-B4, RECCE, CLIP) | **sekali**, skip otomatis |
| 3 | Download DF40 training set ke Drive | **sekali**, skip otomatis |
| 4 | Verifikasi dataset layout (FF++, Celeb-DF di Drive user) | setiap session |
| 5 | Face extraction (gen1 FF++, gen2 Celeb-DF) + catalog DF40 (gen3) | **sekali**, skip otomatis |
| 6 | Splits + soft labels | **sekali**, skip otomatis |
| 7 | Copy hot data ke disk lokal | setiap session (cepat di-/content/) |
| 8 | Training: gen1 initial distillation | 1x per eksperimen |
| 9 | Training: gen2 continual (replay) | 1x per eksperimen |
| 10 | Training: gen3 continual (replay) | 1x per eksperimen |
| 11 | Edge evaluation (TFLite fp32/fp16/int8) | setelah training |
| 12 | Generate figures | terakhir |

## Asumsi Drive Layout

```
/content/drive/MyDrive/CKD_Thesis/
├── checkpoints/teachers/                # diisi Bagian 2
├── datasets/raw/
│   ├── faceforensics_pp/                # <- kamu download manual sekali
│   │   ├── original_sequences/youtube/c23/videos/*.mp4
│   │   └── manipulated_sequences/<teknik>/c23/videos/*.mp4
│   ├── celeb_df_v2/                     # <- kamu download manual sekali
│   │   ├── Celeb-real/*.mp4
│   │   ├── YouTube-real/*.mp4
│   │   └── Celeb-synthesis/*.mp4
│   └── df40/                            # diisi Bagian 3 (gdown)
│       └── <technique>/<ff|cdf>/<video_id>/*.png
├── datasets/faces/                      # diisi Bagian 5
├── datasets/splits/                     # diisi Bagian 6
├── soft_labels/                         # diisi Bagian 6
├── checkpoints/students/                # diisi Bagian 8-10
├── edge/                                # diisi Bagian 11
└── results/                             # diisi Bagian 11-12
```

> FF++ dan Celeb-DF diunduh manual karena butuh signed EULA / form persetujuan.

## 1. Setup — Mount Drive, Clone, Install

In [ ]:
# 1.1 Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/CKD_Thesis'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

In [ ]:
# 1.2 Clone / pull repo
GITHUB_USER = 'tesisbright123-blip'
REPO_NAME   = 'ckd-deepfake'
REPO_URL    = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'
REPO_DIR    = f'/content/{REPO_NAME}'

if os.path.isdir(REPO_DIR):
    %cd {REPO_DIR}
    !git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git log -1 --oneline

In [ ]:
# 1.3 Install dependencies (idempotent; pip akan skip yang sudah terpasang)
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q gdown

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
print(f'PyTorch {torch.__version__}  CUDA={torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: GPU tidak terdeteksi. Runtime > Change runtime type > T4 GPU')

## 2. Download Teacher Weights

| Model | Sumber | Ukuran |
|-------|--------|--------|
| EfficientNet-B4 | DeepfakeBench GitHub Releases | ~68 MB |
| RECCE | DeepfakeBench GitHub Releases | ~183 MB |
| CLIP ViT-L/14 (CLIPping) | Google Drive | ~1.7 GB |

Total ~2 GB. Skip otomatis kalau file sudah ada.

In [ ]:
TEACHER_DIR = os.path.join(DRIVE_ROOT, 'checkpoints', 'teachers')
os.makedirs(TEACHER_DIR, exist_ok=True)

# EfficientNet-B4
effnb4 = os.path.join(TEACHER_DIR, 'efficientnet_b4.pth')
if os.path.isfile(effnb4):
    print(f'[skip] efficientnet_b4.pth ({os.path.getsize(effnb4)/1e6:.1f} MB)')
else:
    !wget -q --show-progress \
        'https://github.com/SCLBD/DeepfakeBench/releases/download/v1.0.1/effnb4_best.pth' \
        -O '{effnb4}'

# RECCE
recce = os.path.join(TEACHER_DIR, 'recce.pth')
if os.path.isfile(recce):
    print(f'[skip] recce.pth ({os.path.getsize(recce)/1e6:.1f} MB)')
else:
    !wget -q --show-progress \
        'https://github.com/SCLBD/DeepfakeBench/releases/download/v1.0.1/recce_best.pth' \
        -O '{recce}'

# CLIP ViT-L/14 (CLIPping)
clip = os.path.join(TEACHER_DIR, 'clip_clipping.pth')
if os.path.isfile(clip):
    print(f'[skip] clip_clipping.pth ({os.path.getsize(clip)/1e6:.1f} MB)')
else:
    import gdown
    gdown.download(id='1jtkBoIuMw5wrooTv-anh668G3_Xt-UnX', output=clip, quiet=False, fuzzy=True)

print('\n=== Teacher weights ===')
for fname in ['efficientnet_b4.pth', 'recce.pth', 'clip_clipping.pth']:
    fpath = os.path.join(TEACHER_DIR, fname)
    tag = f'{os.path.getsize(fpath)/1e6:.1f} MB' if os.path.isfile(fpath) else 'MISSING'
    print(f'  {fname:<28}  {tag}')

## 3. Download DF40 (gen3 fake dataset)

DF40 hanya tersedia di Google Drive dari author ([YZY-stack/DF40](https://github.com/YZY-stack/DF40)).
Notebook ini download folder training (~50 GB, FF++ domain) via `gdown.download_folder`. Skip otomatis kalau folder sudah ada.

> **Catatan:** Download pertama bisa 1–3 jam tergantung bandwidth Colab. `gdown` resume-safe — kalau putus tengah jalan, jalankan ulang cell ini.

> **Hemat ruang:** Kalau Drive kamu terbatas, edit variable `DF40_TECHNIQUES_KEEP` di bawah — hanya folder teknik yang namanya match yang akan di-catalog. (Download tetap ambil semua; filtering saat catalog.)

In [ ]:
DF40_DRIVE_FOLDER_ID = '1U8meBbqVvmUkc5GD0jxct6xe6Gwk9wKD'  # DF40 training (FF++ domain, ~50 GB)
DF40_DST = os.path.join(DRIVE_ROOT, 'datasets', 'raw', 'df40')
os.makedirs(DF40_DST, exist_ok=True)

already = [p for p in os.listdir(DF40_DST) if os.path.isdir(os.path.join(DF40_DST, p))]
if already:
    print(f'[skip] DF40 folder sudah berisi {len(already)} subfolder: {already[:6]}...')
else:
    import gdown
    gdown.download_folder(
        id=DF40_DRIVE_FOLDER_ID,
        output=DF40_DST,
        quiet=False,
        use_cookies=False,
    )
    print(f'\nDF40 di {DF40_DST}')

!du -sh '{DF40_DST}' 2>/dev/null || true

## 4. Verifikasi Dataset Layout

Cek folder FF++ + Celeb-DF yang **kamu upload manual** ke Drive. Kalau folder kosong, sel Bagian 5 akan error jelas dan kasih tahu path mana yang missing.

In [ ]:
from pathlib import Path

RAW = Path(DRIVE_ROOT) / 'datasets' / 'raw'
CHECKS = {
    'faceforensics_pp/original_sequences/youtube/c23/videos': RAW / 'faceforensics_pp' / 'original_sequences' / 'youtube' / 'c23' / 'videos',
    'faceforensics_pp/manipulated_sequences':                  RAW / 'faceforensics_pp' / 'manipulated_sequences',
    'celeb_df_v2/Celeb-real':                                  RAW / 'celeb_df_v2' / 'Celeb-real',
    'celeb_df_v2/Celeb-synthesis':                             RAW / 'celeb_df_v2' / 'Celeb-synthesis',
    'df40':                                                    RAW / 'df40',
}

def _count(p: Path, pattern: str) -> int:
    return sum(1 for _ in p.rglob(pattern)) if p.is_dir() else 0

for label, path in CHECKS.items():
    if not path.is_dir():
        print(f'  [MISSING] {label:<55}  {path}')
        continue
    mp4 = _count(path, '*.mp4')
    img = _count(path, '*.png') + _count(path, '*.jpg')
    print(f'  [OK]      {label:<55}  mp4={mp4} img={img}')

## 5. Face Extraction + DF40 Catalog

- `scripts/01` extract face JPEGs dari video FF++ (gen1) dan Celeb-DF (gen2).
- `scripts/01b` catalog DF40 images (sudah face-cropped, tidak butuh ekstraksi) + pinjam row REAL dari `gen1` (FF++ real).

Skip otomatis kalau `metadata_<gen>.csv` sudah ada.

In [ ]:
FACES_DIR = Path(DRIVE_ROOT) / 'datasets' / 'faces'

def _already_have(gen: str) -> bool:
    return (FACES_DIR / f'metadata_{gen}.csv').is_file()

# gen1 — FF++ face extraction
if _already_have('gen1'):
    print('[skip] metadata_gen1.csv sudah ada')
else:
    !python scripts/01_extract_faces.py --config configs/default.yaml --generation gen1

# gen2 — Celeb-DF face extraction
if _already_have('gen2'):
    print('[skip] metadata_gen2.csv sudah ada')
else:
    !python scripts/01_extract_faces.py --config configs/default.yaml --generation gen2

# gen3 — DF40 catalog (no face extraction, borrow real from gen1)
if _already_have('gen3'):
    print('[skip] metadata_gen3.csv sudah ada')
else:
    !python scripts/01b_catalog_df40.py --config configs/default.yaml --borrow-real-from gen1

!ls -la '{FACES_DIR}'/metadata_*.csv 2>/dev/null

## 6. Splits + Soft Labels

- `scripts/02` → video-level 70/15/15 stratified splits per generation.
- `scripts/03` → ensemble soft-labels dari 3 teacher untuk train split.

In [ ]:
SPLITS_DIR = Path(DRIVE_ROOT) / 'datasets' / 'splits'
SOFT_DIR   = Path(DRIVE_ROOT) / 'soft_labels'

for gen in ('gen1', 'gen2', 'gen3'):
    if (SPLITS_DIR / f'{gen}_train.csv').is_file():
        print(f'[skip] splits {gen} sudah ada')
        continue
    !python scripts/02_generate_splits.py --config configs/default.yaml --generation {gen}

for gen in ('gen1', 'gen2', 'gen3'):
    # scripts/03 writes to {drive}/soft_labels/{gen}/{split}/ensemble.npy
    if (SOFT_DIR / gen / 'train' / 'ensemble.npy').is_file():
        print(f'[skip] soft labels {gen} sudah ada')
        continue
    !python scripts/03_generate_soft_labels.py --config configs/default.yaml --generation {gen}

!ls -la '{SPLITS_DIR}' 2>/dev/null
!find '{SOFT_DIR}' -name 'ensemble.npy' 2>/dev/null

## 7. Copy Hot Data ke Disk Lokal

Drive I/O lambat saat loader banyak worker. Copy `datasets/` + `soft_labels/` ke `/content/ckd_local/` sekali di awal session (I/O 10–20x lebih cepat). Idempotent.

In [ ]:
import shutil, time
LOCAL_ROOT = '/content/ckd_local'
os.makedirs(LOCAL_ROOT, exist_ok=True)

for sub in ('datasets', 'soft_labels'):
    src = os.path.join(DRIVE_ROOT, sub)
    dst = os.path.join(LOCAL_ROOT, sub)
    if not os.path.isdir(src):
        print(f'[skip] {src} belum ada')
        continue
    if os.path.isdir(dst):
        print(f'[skip] {dst} sudah ada')
        continue
    t0 = time.time()
    print(f'[copy] {src} -> {dst}')
    shutil.copytree(src, dst)
    print(f'  selesai {time.time()-t0:.1f}s')

!du -sh '{LOCAL_ROOT}'/* 2>/dev/null

## 8. Training gen1 — Initial Distillation

Student = MobileNetV3-Large dari scratch (ImageNet init). Loss = α·CE + (1-α)·KD(T) dari ensemble 3 teachers.

Output: `{drive}/checkpoints/students/gen1/best.pth` + metrics JSON.

In [ ]:
CHECKPOINT_GEN1 = Path(DRIVE_ROOT) / 'checkpoints' / 'students' / 'gen1' / 'best.pth'
if CHECKPOINT_GEN1.is_file():
    print(f'[skip] {CHECKPOINT_GEN1} sudah ada. Hapus kalau mau re-train.')
else:
    !python scripts/04_initial_distillation.py --config configs/default.yaml --generation gen1

## 9. Training gen2 — Continual Distillation (replay)

Loss = α·CE + β·KD(gen2 teachers) + γ·KD(gen1 teachers via replay buffer). Anti-forgetting method = **replay** (default dari config).

Ganti `--method ewc` atau `--method lwf` untuk ablasi.

In [ ]:
METHOD = 'replay'  # ewc | lwf | replay
CHECKPOINT_GEN2 = Path(DRIVE_ROOT) / 'checkpoints' / 'students' / f'gen2_{METHOD}' / 'best.pth'
if CHECKPOINT_GEN2.is_file():
    print(f'[skip] {CHECKPOINT_GEN2} sudah ada.')
else:
    !python scripts/05_continual_distillation.py \
        --config configs/default.yaml \
        --generation gen2 \
        --method {METHOD} \
        --previous-checkpoint '{CHECKPOINT_GEN1}'

## 10. Training gen3 — Continual Distillation (replay)

In [ ]:
CHECKPOINT_GEN3 = Path(DRIVE_ROOT) / 'checkpoints' / 'students' / f'gen3_{METHOD}' / 'best.pth'
if CHECKPOINT_GEN3.is_file():
    print(f'[skip] {CHECKPOINT_GEN3} sudah ada.')
else:
    !python scripts/05_continual_distillation.py \
        --config configs/default.yaml \
        --generation gen3 \
        --method {METHOD} \
        --previous-checkpoint '{CHECKPOINT_GEN2}'

## 11. Edge Evaluation (TFLite)

Export student gen3 → ONNX → TFLite dalam 3 mode (fp32, fp16, int8 dengan kalibrasi), lalu benchmark latency + accuracy pada test split.

In [ ]:
EDGE_JSON = Path(DRIVE_ROOT) / 'results' / 'raw' / f'gen3_{METHOD}_edge_metrics.json'
if EDGE_JSON.is_file():
    print(f'[skip] {EDGE_JSON} sudah ada.')
else:
    !python scripts/07_edge_evaluation.py \
        --config configs/default.yaml \
        --generation gen3 \
        --method {METHOD} \
        --modes fp32,fp16,int8 \
        --num-runs 100 \
        --num-threads 1

## 12. Generate Figures

Auto-discover semua JSON di `{drive}/results/raw/` dan emit PNG + PDF (300 DPI, publication-ready) ke `{drive}/results/figures/`.

In [ ]:
!python scripts/08_generate_figures.py --config configs/default.yaml --figures all

FIG_DIR = Path(DRIVE_ROOT) / 'results' / 'figures'
!ls -la '{FIG_DIR}' 2>/dev/null

## Selesai

Artifacts di Drive:

- `checkpoints/students/gen{1,2,3}*/best.pth` — bobot student
- `results/raw/*_{initial,continual,edge}_metrics.json` — metric mentah
- `results/figures/*.png` + `*.pdf` — figure untuk tesis
- `edge/gen3_{method}/tflite/*.tflite` — artefak deployable

Untuk ablation, jalankan manual setelah pipeline utama selesai:

```bash
!python scripts/06_ablation_study.py --ablation A2 --config configs/default.yaml   # temperature sweep
!python scripts/06_ablation_study.py --ablation A3 --config configs/default.yaml   # buffer size
```